# Clase 13 — IA aplicada a problemas biológicos y químicos · **versión resuelta**

**AIG4B — Inteligencia Artificial Generativa Aplicada en Biomedicina (Universidad Austral)**
**Profesores: Marco Sanchez Sorondo y Paulo Veiga**

Hoy: del plegamiento de proteínas (AlphaFold) a la generación de moléculas. Tres secciones prácticas:
1. **GNNs para propiedades moleculares**: predecir si una droga cruza la barrera hematoencefálica.
2. **Generación molecular: VAE vs Difusión Latente**: comparar dos paradigmas generativos sobre moléculas.
3. **ESMFold**: predecir estructura 3D de proteínas desde la secuencia, sin MSA.


## Setup

Instalación condicional para Colab. Si corrés local con todo instalado, esta celda no hace nada.


In [ ]:
import sys, subprocess
def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import torch_geometric  # noqa
except ImportError:
    _pip("torch_geometric")
try:
    import rdkit  # noqa
except ImportError:
    _pip("rdkit")
try:
    import py3Dmol  # noqa
except ImportError:
    _pip("py3Dmol")
try:
    import transformers  # noqa
except ImportError:
    _pip("transformers")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # silenciar errores de parseo de SMILES inválidos
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

torch.manual_seed(42)
np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  PyG: {torch_geometric.__version__}")


---

## Sección 1 — GNNs para propiedades moleculares

### Motivación: la barrera hematoencefálica

La **barrera hematoencefálica (BBB)** es un filtro biológico que protege el cerebro de toxinas y patógenos en la sangre. El problema: también bloquea la mayoría de los fármacos. Diseñar drogas para enfermedades del SNC (Alzheimer, Parkinson, depresión, dolor crónico) requiere predecir si una molécula cruza la BBB.

El dataset **BBBP** (MoleculeNet, ~2050 moléculas) etiqueta cada SMILES con `1` (cruza) o `0` (no cruza). Vamos a entrenar un GNN para predecirlo.


In [ ]:
# Descargar dataset BBBP
import urllib.request, os
BBBP_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
if not os.path.exists("BBBP.csv"):
    urllib.request.urlretrieve(BBBP_URL, "BBBP.csv")
df = pd.read_csv("BBBP.csv")
print(df.shape)
df.head(3)


In [ ]:
# Filtrar SMILES válidos
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)
print(f"{len(df)} moléculas válidas")
print(f"Clase 1 (cruza BBB): {df['p_np'].sum()} ({df['p_np'].mean()*100:.1f}%)")


### Visualización: 4 moléculas


In [ ]:
samples_pos = df[df["p_np"] == 1].sample(2, random_state=0)
samples_neg = df[df["p_np"] == 0].sample(2, random_state=0)
mols = list(samples_pos["mol"]) + list(samples_neg["mol"])
labels = ["cruza BBB"] * 2 + ["NO cruza"] * 2
img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(300, 200), legends=labels)
img


### Featurización: SMILES → grafo PyG

Cada átomo es un nodo con features (elemento, grado, carga, hibridación, aromaticidad), cada enlace es una arista.


In [ ]:
ATOM_FEATURES = {
    'atomic_num': [6, 7, 8, 9, 15, 16, 17, 35, 53],  # C N O F P S Cl Br I
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2],
    'hybridization': [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                      Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                      Chem.rdchem.HybridizationType.SP3D2],
}

def onek(x, allowed):
    if x not in allowed:
        x = allowed[-1]
    return [int(x == a) for a in allowed]

def atom_features(atom):
    return (onek(atom.GetAtomicNum(), ATOM_FEATURES['atomic_num'])
            + onek(atom.GetDegree(), ATOM_FEATURES['degree'])
            + onek(atom.GetFormalCharge(), ATOM_FEATURES['formal_charge'])
            + onek(atom.GetHybridization(), ATOM_FEATURES['hybridization'])
            + [int(atom.GetIsAromatic())])

def mol_to_graph(mol, y):
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty(2, 0, dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=torch.tensor([y], dtype=torch.float))

graphs = [mol_to_graph(m, y) for m, y in zip(df["mol"], df["p_np"])]
print(f"Features por átomo: {graphs[0].x.shape[1]}")
print(f"Total grafos: {len(graphs)}")


In [ ]:
# Split train/val/test estratificado 70/15/15
from sklearn.model_selection import train_test_split
labels = df["p_np"].values
idx_train, idx_temp = train_test_split(range(len(graphs)), test_size=0.30, stratify=labels, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=labels[idx_temp], random_state=42)
train_set = [graphs[i] for i in idx_train]
val_set = [graphs[i] for i in idx_val]
test_set = [graphs[i] for i in idx_test]
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)


---

### Ejercicio 1.1 — Implementar y entrenar una GNN sobre BBBP

Implementá una GNN con esta arquitectura:

| Capa | Detalle |
|---|---|
| `GCNConv` × 3 | hidden 64, con `BatchNorm1d` + `ReLU` entre capas |
| `global_mean_pool` | reduce el grafo a un vector |
| MLP head | `Linear(64, 32)` → `ReLU` → `Dropout(0.3)` → `Linear(32, 1)` |

Entrenala 30 epochs con `BCEWithLogitsLoss` + `Adam(lr=1e-3)`. Reportá **accuracy**, **AUC** y **matriz de confusión** sobre el test set.


In [ ]:
# === SOLUCIÓN ===
class GCN(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn3 = nn.BatchNorm1d(hidden)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1)
        )
    def forward(self, x, edge_index, batch):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.relu(self.bn3(self.conv3(h, edge_index)))
        h = global_mean_pool(h, batch)
        return self.head(h).squeeze(-1)

in_dim = graphs[0].x.shape[1]
model = GCN(in_dim).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# === SOLUCIÓN ===
@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, ps = [], []
    for data in loader:
        data = data.to(DEVICE)
        logits = model(data.x, data.edge_index, data.batch)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(data.y.cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return y, p

# Training
for epoch in range(30):
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(DEVICE)
        opt.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = loss_fn(out, data.y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * data.num_graphs
    if (epoch+1) % 5 == 0:
        y_val, p_val = evaluate(val_loader)
        auc_val = roc_auc_score(y_val, p_val)
        print(f"Epoch {epoch+1:02d}  train_loss={total_loss/len(train_set):.4f}  val_AUC={auc_val:.4f}")


In [ ]:
# === SOLUCIÓN ===
# Evaluación final sobre test
y_test, p_test = evaluate(test_loader)
pred = (p_test > 0.5).astype(int)
acc = (pred == y_test).mean()
auc = roc_auc_score(y_test, p_test)
cm = confusion_matrix(y_test.astype(int), pred)

print(f"Test accuracy: {acc:.4f}")
print(f"Test AUC: {auc:.4f}")
print(f"Matriz de confusión:")
print(cm)

fig, ax = plt.subplots(1, 1, figsize=(4, 3.5))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='red', fontsize=14)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['no cruza', 'cruza'])
ax.set_yticklabels(['no cruza', 'cruza'])
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (BBBP test)')
plt.tight_layout()
plt.show()


### Preguntas conceptuales — Ejercicio 1.1

Respondé brevemente cada una. No hay una sola respuesta correcta; lo que evaluamos es que justifiques.

1. **¿Por qué message passing (GCN) y no un MLP sobre fingerprints (vector fijo)?**
2. **¿Por qué `global_mean_pool` después de las capas GCN? ¿Por qué no `flatten`?**
3. **¿Por qué reportamos AUC además de accuracy en este dataset?**
4. **¿Qué pasaría si pongo 10 capas GCN en vez de 3?** (pista: oversmoothing)
5. **¿Por qué `BatchNorm` entre capas GCN y no `LayerNorm`?**


<!-- SOLUCIÓN -->

**Respuestas — Ejercicio 1.1**

1. **GCN vs MLP sobre fingerprints**: los fingerprints (Morgan/ECFP) son una representación fija que codifica subestructuras pre-definidas. Pierde información cuando los radios no capturan el patrón relevante. Las GCN aprenden representaciones específicas para la tarea — el message passing permite que cada átomo "vea" información de sus vecinos según el problema.

2. **`global_mean_pool` vs `flatten`**: las moléculas tienen distinto número de átomos (no se pueden flattenear a un vector fijo). El pooling agrega los embeddings nodales en un único vector de tamaño fijo, **invariante a permutaciones** de los átomos (que es la propiedad correcta — el orden de los átomos en SMILES es arbitrario).

3. **AUC además de accuracy**: BBBP está desbalanceado (~76% positivos). Un modelo que predice "cruza" siempre logra ~76% accuracy. AUC mide la capacidad discriminativa independientemente del umbral, así que no se deja engañar por el desbalance.

4. **10 capas GCN — oversmoothing**: con más capas, los embeddings de todos los nodos se vuelven cada vez más similares entre sí (cada uno termina viendo casi toda la molécula). Resultado: el modelo pierde discriminación entre moléculas. 3-5 capas suele ser el sweet spot para grafos moleculares.

5. **BatchNorm vs LayerNorm**: en GNNs sobre grafos pequeños/variables, BatchNorm sobre el conjunto de nodos del batch funciona bien porque hay muchas instancias de nodos. LayerNorm normaliza por nodo individualmente y suele ser preferido en transformers donde el batch es chico. La elección no es absoluta; ambas funcionan, pero BatchNorm es más estándar en GCNs.


---

## Sección 2 — Generación molecular: VAE vs Difusión Latente

### Motivación

El espacio químico de moléculas drug-like tiene unas **10⁶⁰** estructuras posibles. Imposible enumerarlo. Necesitamos **generar** moléculas dirigidas: que cumplan propiedades deseadas, que sean válidas, que sean diversas. En esta sección comparamos dos paradigmas:

- **VAE de SMILES** (paradigma clásico, 2018) — actúa como baseline.
- **Difusión latente** (paradigma actual) — el modelo protagonista.

> **Conexión con Clase 9**: lo que vas a hacer acá es **exactamente el mismo paradigma de Stable Diffusion**, pero sobre moléculas en vez de imágenes. Stable Diffusion = difusión sobre el espacio latente de un VAE de imágenes. Acá: difusión sobre el espacio latente de un VAE de moléculas.

### Dataset

Subset de **ZINC** (10k moléculas drug-like, longitud ≤ 50 caracteres SMILES). Lo curamos desde la fuente original (ZINC 250k del repo `aspuru-guzik-group/chemical_vae`).


In [ ]:
import os, random, urllib.request

ZINC_RAW = "zinc_250k_raw.csv"
ZINC_PATH = "zinc_subset_10k.txt"

if not os.path.exists(ZINC_PATH):
    if not os.path.exists(ZINC_RAW):
        print("Descargando ZINC 250k...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/models/zinc/250k_rndm_zinc_drugs_clean_3.csv",
            ZINC_RAW
        )
    print("Filtrando a subset de 10k...")
    raw_df = pd.read_csv(ZINC_RAW)
    smiles = raw_df["smiles"].astype(str).str.strip().tolist()
    filtered = [s for s in smiles if 5 <= len(s) <= 50]
    random.seed(42)
    random.shuffle(filtered)
    subset = filtered[:10000]
    with open(ZINC_PATH, "w") as f:
        for s in subset:
            f.write(s + "\n")
    print(f"Subset escrito: {len(subset)} SMILES")

with open(ZINC_PATH) as f:
    zinc_smiles = [s.strip() for s in f if s.strip()]
print(f"{len(zinc_smiles)} moléculas de ZINC cargadas")
print("Ejemplos:", zinc_smiles[:3])


### Arquitectura del VAE de SMILES

LSTM encoder bidireccional + LSTM decoder autoregresivo, latente de 64 dimensiones. Char-level sobre el alfabeto SMILES.


In [ ]:
# Configuración del VAE
EMB_DIM, HIDDEN, LATENT, MAX_LEN = 64, 256, 64, 52

# Vocabulario char-level
def build_vocab(smiles_list):
    chars = set()
    for s in smiles_list:
        chars.update(s)
    vocab = ["<pad>", "<start>", "<end>"] + sorted(chars)
    return {c: i for i, c in enumerate(vocab)}, vocab

VOCAB, ITOS = build_vocab(zinc_smiles)
print(f"Vocab size: {len(VOCAB)}")

def encode_smiles(s, max_len=MAX_LEN):
    ids = [VOCAB["<start>"]] + [VOCAB[c] for c in s if c in VOCAB] + [VOCAB["<end>"]]
    ids = ids + [VOCAB["<pad>"]] * (max_len - len(ids))
    return ids[:max_len]


In [ ]:
# Arquitectura VAE
class VAEEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMB_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMB_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.mu = nn.Linear(HIDDEN * 2, LATENT)
        self.logvar = nn.Linear(HIDDEN * 2, LATENT)
    def forward(self, x):
        e = self.emb(x)
        _, (h, _) = self.lstm(e)
        h = torch.cat([h[0], h[1]], dim=-1)
        return self.mu(h), self.logvar(h)

class VAEDecoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMB_DIM, padding_idx=0)
        self.latent_to_hidden = nn.Linear(LATENT, HIDDEN)
        self.lstm = nn.LSTM(EMB_DIM + LATENT, HIDDEN, batch_first=True)
        self.out = nn.Linear(HIDDEN, vocab_size)
    def forward(self, z, target):
        B, L = target.shape
        e = self.emb(target)
        z_rep = z.unsqueeze(1).expand(-1, L, -1)
        x = torch.cat([e, z_rep], dim=-1)
        h0 = torch.tanh(self.latent_to_hidden(z)).unsqueeze(0)
        c0 = torch.zeros_like(h0)
        out, _ = self.lstm(x, (h0, c0))
        return self.out(out)

class SmilesVAE(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = VAEEncoder(vocab_size)
        self.decoder = VAEDecoder(vocab_size)
    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        return mu + std * torch.randn_like(std)
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        target_in = x[:, :-1]
        logits = self.decoder(z, target_in)
        target_out = x[:, 1:]
        return logits, target_out, mu, logvar

vae = SmilesVAE(len(VOCAB)).to(DEVICE)
print(f"VAE instanciado. Parámetros: {sum(p.numel() for p in vae.parameters()):,}")


### Entrenamiento del VAE

> **Importante**: usá GPU (Runtime → Change runtime type → GPU en Colab). Con GPU tarda ~5-8 min. En CPU es demasiado lento (>1h).

El loop usa **free bits** (cota mínima de KL por dimensión) para evitar **posterior collapse** — un problema clásico de VAEs con decoders autoregresivos fuertes. Sin free bits, el decoder ignora el latente.


In [ ]:
# Loss con free bits + KL annealing
def vae_loss(logits, target, mu, logvar, beta, free_bits=0.5):
    recon = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1), ignore_index=0)
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)
    kl = kl_per_dim.sum(dim=1).mean()
    return recon + beta * kl, recon.item(), kl.item()

# Dataset y entrenamiento
encoded = torch.tensor([encode_smiles(s) for s in zinc_smiles], dtype=torch.long)
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset
vae_ds = TensorDataset(encoded)
vae_dl = TorchDataLoader(vae_ds, batch_size=256, shuffle=True, drop_last=True)

opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)
EPOCHS_VAE = 25

for epoch in range(EPOCHS_VAE):
    beta = min(1.0, epoch / 8.0)
    vae.train()
    total, total_recon, total_kl = 0, 0, 0
    for (x,) in vae_dl:
        x = x.to(DEVICE)
        logits, target, mu, logvar = vae(x)
        loss, r, k = vae_loss(logits, target, mu, logvar, beta)
        opt_vae.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        opt_vae.step()
        total += loss.item(); total_recon += r; total_kl += k
    n = len(vae_dl)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS_VAE}  recon={total_recon/n:.4f}  kl={total_kl/n:.4f}  beta={beta:.2f}")

vae.eval()
print("VAE entrenado.")


In [ ]:
# Helper de muestreo del VAE (autoregresivo desde z)
@torch.no_grad()
def decode_z(z, temperature=1.0):
    z = z.to(DEVICE)
    B = z.size(0)
    h0 = torch.tanh(vae.decoder.latent_to_hidden(z)).unsqueeze(0)
    c0 = torch.zeros_like(h0)
    prev = torch.full((B, 1), VOCAB["<start>"], dtype=torch.long, device=DEVICE)
    out_ids = [[] for _ in range(B)]
    done = [False] * B
    for _ in range(MAX_LEN - 1):
        e = vae.decoder.emb(prev)
        x = torch.cat([e, z.unsqueeze(1)], dim=-1)
        o, (h0, c0) = vae.decoder.lstm(x, (h0, c0))
        logits = vae.decoder.out(o[:, -1, :])
        probs = torch.softmax(logits / temperature, dim=-1)
        tok = torch.multinomial(probs, 1)
        for i in range(B):
            if done[i]:
                continue
            t = tok[i, 0].item()
            if t == VOCAB["<end>"] or t == VOCAB["<pad>"]:
                done[i] = True
            else:
                out_ids[i].append(t)
        prev = tok
        if all(done):
            break
    return ["".join(ITOS[i] for i in seq) for seq in out_ids]

def sample_vae(n=2000, temperature=0.7):
    # Nota: temperatura 0.7 es estándar para SMILES VAE char-level
    # (los modelos chicos con MAX_LEN=52 son ruidosos a temp=1.0).
    z = torch.randn(n, LATENT)
    return decode_z(z, temperature)


### Métricas: validez, unicidad, novedad

Usaremos las **mismas 3 métricas** para VAE y para difusión, permitiendo comparación directa.


In [ ]:
train_set_canon = set()
for s in zinc_smiles:
    m = Chem.MolFromSmiles(s)
    if m is not None:
        train_set_canon.add(Chem.MolToSmiles(m))

def evaluate_generation(smiles_list, ref_set=train_set_canon, label=""):
    n = len(smiles_list)
    valid_canon = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if m is not None:
            valid_canon.append(Chem.MolToSmiles(m))
    pct_valid = len(valid_canon) / n * 100
    unique = set(valid_canon)
    pct_unique = len(unique) / max(len(valid_canon), 1) * 100
    novel = unique - ref_set
    pct_novel = len(novel) / max(len(unique), 1) * 100
    print(f"{label}: {pct_valid:.1f}% válidas | {pct_unique:.1f}% únicas (entre válidas) | {pct_novel:.1f}% nuevas (no en train)")
    return {"valid": pct_valid, "unique": pct_unique, "novel": pct_novel}

# Baseline VAE
vae_samples = sample_vae(n=2000)
print("Ejemplos VAE:", vae_samples[:5])
metrics_vae = evaluate_generation(vae_samples, label="VAE")


---

### Ejercicio 2.1 — Implementar difusión latente sobre el espacio del VAE

**Idea**: en vez de muestrear `z ~ N(0, I)` ciegamente del VAE (mucho del espacio latente decodifica a basura), entrenamos un modelo de difusión que aprende a generar latentes "buenos" — los que el decoder sabe transformar en SMILES válidos.

**Pasos**:
1. Codificá todo el train set al espacio latente: `μ = encoder(x)` para cada SMILES → matriz `(N, 64)`.
2. Implementá un schedule de ruido (lineal, 200 pasos).
3. Implementá un denoiser: MLP de 4 capas que recibe `(z_noisy, t)` y predice el ruido.
4. Entrenalo con MSE entre ruido predicho y ruido real.
5. Sampleá: ruido inicial `~ N(0, I)` → denoising iterativo → decoder del VAE → SMILES.
6. Calculá las mismas 3 métricas. Compará en una tabla.


In [ ]:
# === SOLUCIÓN ===
# 1) Codificar el train set al espacio latente del VAE
@torch.no_grad()
def encode_smiles_batch(smiles_list, batch_size=128):
    latents = []
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        encoded = []
        for s in batch:
            ids = [VOCAB["<start>"]] + [VOCAB[c] for c in s if c in VOCAB] + [VOCAB["<end>"]]
            ids = ids + [VOCAB["<pad>"]] * (MAX_LEN - len(ids))
            encoded.append(ids[:MAX_LEN])
        x = torch.tensor(encoded, dtype=torch.long, device=DEVICE)
        mu, _ = vae.encoder(x)
        latents.append(mu.cpu())
    return torch.cat(latents, dim=0)

train_latents = encode_smiles_batch(zinc_smiles)
print(f"Latentes del train: {train_latents.shape}  |  mean={train_latents.mean():.3f}  std={train_latents.std():.3f}")


In [ ]:
# === SOLUCIÓN ===
# 2) Schedule de ruido lineal (todo en DEVICE)
T = 200
betas = torch.linspace(1e-4, 0.02, T, device=DEVICE)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_acp = torch.sqrt(alphas_cumprod)
sqrt_1macp = torch.sqrt(1 - alphas_cumprod)

def add_noise(z0, t):
    # z0: (B, D), t: (B,) — ambos en DEVICE
    noise = torch.randn_like(z0)
    s_acp = sqrt_acp[t].unsqueeze(1)
    s_1macp = sqrt_1macp[t].unsqueeze(1)
    z_t = s_acp * z0 + s_1macp * noise
    return z_t, noise

# 3) Denoiser: MLP con embedding de timestep
class TimestepEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

class Denoiser(nn.Module):
    def __init__(self, latent_dim=64, hidden=256, t_emb=128):
        super().__init__()
        self.t_emb = TimestepEmb(t_emb)
        self.t_proj = nn.Sequential(nn.Linear(t_emb, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.in_proj = nn.Linear(latent_dim, hidden)
        self.mid = nn.Sequential(
            nn.SiLU(), nn.Linear(hidden, hidden),
            nn.SiLU(), nn.Linear(hidden, hidden),
            nn.SiLU(), nn.Linear(hidden, hidden),
        )
        self.out_proj = nn.Linear(hidden, latent_dim)
    def forward(self, z, t):
        h = self.in_proj(z)
        te = self.t_proj(self.t_emb(t))
        h = h + te
        h = self.mid(h)
        return self.out_proj(h)

denoiser = Denoiser(latent_dim=LATENT).to(DEVICE)
print(f"Parámetros denoiser: {sum(p.numel() for p in denoiser.parameters()):,}")


In [ ]:
# === SOLUCIÓN ===
# 4) Entrenamiento
opt_d = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
train_latents_dev = train_latents.to(DEVICE)
N = train_latents_dev.size(0)
BATCH = 256
EPOCHS_D = 30

for epoch in range(EPOCHS_D):
    perm = torch.randperm(N, device=DEVICE)
    total = 0
    for i in range(0, N, BATCH):
        idx = perm[i:i+BATCH]
        z0 = train_latents_dev[idx]
        t = torch.randint(0, T, (z0.size(0),), device=DEVICE)
        z_t, noise = add_noise(z0, t)
        pred = denoiser(z_t, t)
        loss = F.mse_loss(pred, noise)
        opt_d.zero_grad()
        loss.backward()
        opt_d.step()
        total += loss.item() * z0.size(0)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}  MSE={total/N:.4f}")


In [ ]:
# === SOLUCIÓN ===
# 5) Sampleo: denoising iterativo
@torch.no_grad()
def sample_diffusion(n=2000):
    z = torch.randn(n, LATENT, device=DEVICE)
    for t in reversed(range(T)):
        t_batch = torch.full((n,), t, device=DEVICE, dtype=torch.long)
        eps_pred = denoiser(z, t_batch)
        alpha_t = alphas[t]
        acp_t = alphas_cumprod[t]
        coef = (1 - alpha_t) / torch.sqrt(1 - acp_t)
        z = (1 / torch.sqrt(alpha_t)) * (z - coef * eps_pred)
        if t > 0:
            sigma_t = torch.sqrt(betas[t])
            z = z + sigma_t * torch.randn_like(z)
    return decode_z(z, temperature=0.7)

diff_samples = sample_diffusion(n=2000)
print("Ejemplos Difusión:", diff_samples[:5])
metrics_diff = evaluate_generation(diff_samples, label="Difusión Latente")


In [ ]:
# === SOLUCIÓN ===
# 6) Tabla comparativa
print("\n=== COMPARACIÓN FINAL ===\n")
print(f"{'Métrica':<20} {'VAE':>10} {'Difusión':>12}")
print("-" * 44)
for k, name in [("valid", "% válidas"), ("unique", "% únicas"), ("novel", "% nuevas")]:
    print(f"{name:<20} {metrics_vae[k]:>9.1f}% {metrics_diff[k]:>11.1f}%")


### Preguntas conceptuales — Ejercicio 2.1

1. **¿Por qué entrenamos la difusión sobre el espacio latente del VAE y no sobre los SMILES directos?**
2. **¿Qué rol cumple el schedule de ruido `betas`? ¿Qué pasa si `T = 10` en vez de `T = 200`?**
3. **¿Por qué la difusión tiende a dar más muestras válidas que muestrear `z ~ N(0, I)` directamente del VAE?**
4. **¿Cuál es el principal trade-off entre VAE y difusión (más allá de validez)?**
5. **¿Cómo extenderías esto a generación condicionada (ej: "moléculas con peso molecular < 300")?**


<!-- SOLUCIÓN -->

**Respuestas — Ejercicio 2.1**

1. **Difusión latente vs sobre SMILES**: los SMILES son strings discretos con sintaxis estricta. Difusión opera naturalmente sobre espacios continuos (no se "interpola" texto). El VAE ya proyecta SMILES a un espacio continuo de 64 dimensiones donde la difusión puede operar. Mismo motivo por el que Stable Diffusion usa un VAE de imágenes: para pasar a un espacio donde la difusión funcione bien.

2. **Schedule de ruido + T pequeño**: el schedule define cómo se destruye gradualmente la información del dato original a lo largo de los `T` pasos. Con `T = 200` la transición de "dato limpio" a "ruido puro" es suave; el modelo aprende una secuencia de mini-correcciones. Con `T = 10`, cada paso es brusco (mucha varianza por paso), el modelo no puede aprender una buena reversión y la calidad de las muestras baja drásticamente.

3. **Difusión es más válida que VAE puro**: el VAE entrena `z` con una loss que combina reconstrucción + KL hacia `N(0, I)`. Pero la distribución real de los `z` de los datos NO es exactamente `N(0, I)`: tiene huecos, modos, sesgos. Cuando muestreás `z ~ N(0, I)` directamente, caés mucho en zonas "vacías" del latente donde el decoder produce basura. La difusión aprende a generar latentes que están en la distribución real de datos → cae en zonas que el decoder sabe manejar.

4. **Trade-off principal**: **velocidad de muestreo**. El VAE genera en 1 forward pass del decoder (ms). La difusión requiere `T` iteraciones del denoiser (200 forwards). En aplicaciones de búsqueda masiva (millones de candidatos), la diferencia importa. Por eso existen variantes (DDIM, distillation) que reducen los pasos.

5. **Generación condicionada**: la forma estándar es **classifier-free guidance** (igual que Stable Diffusion). Se entrena el denoiser recibiendo además un condicionante `c` (peso molecular, en este caso). Durante el entrenamiento, a veces se pasa `c` y a veces se reemplaza por un "vacío" (entrenamiento conjunto incondicional + condicional). En muestreo, se interpola entre ambos para controlar cuánto peso tiene la condición.


---

## Sección 3 — ESMFold: predecir estructura de proteínas desde la secuencia

En las secciones anteriores trabajamos sobre **moléculas pequeñas** (química). Ahora pasamos al lado **biológico** del problema: estructura de proteínas.

**ESMFold** (Meta, 2022) es un modelo de lenguaje de proteínas (transformer, 3 mil millones de parámetros) entrenado sobre ~200 millones de secuencias. A diferencia de AlphaFold 2, **no necesita MSA**: predice estructura solo desde la secuencia primaria. Es ~60x más rápido que AF2 a costa de una pequeña pérdida de precisión.

Vamos a usarlo en inferencia para predecir la estructura 3D de tres proteínas y comparar resultados.

> **Importante**: esta sección descarga un checkpoint de ~6 GB y necesita GPU para correr en tiempo razonable. En CPU es prohibitivo.


In [ ]:
from transformers import AutoTokenizer, EsmForProteinFolding
import py3Dmol

# Cargar ESMFold v1 desde HuggingFace (descarga ~6 GB la primera vez)
esmfold_tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
esmfold_model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
esmfold_model = esmfold_model.to(DEVICE)
esmfold_model.esm = esmfold_model.esm.half()  # half precision para el LLM (ahorra VRAM)
esmfold_model = esmfold_model.eval()
print(f"ESMFold cargado en {DEVICE}. Parámetros totales: {sum(p.numel() for p in esmfold_model.parameters())/1e9:.2f}B")


In [ ]:
# Tres proteínas con tamaños y composiciones estructurales distintas
PROTEINS = {
    "Insulina (precursor humano)": "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN",
    "Lisozima C (huevo de gallina)": "KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL",
    "GFP (Green Fluorescent Protein)": "MASKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK",
}
for name, seq in PROTEINS.items():
    print(f"{name}: {len(seq)} aminoácidos")


### Helper provisto: predecir estructura + visualización

`predict_structure` corre el modelo y devuelve el PDB como string + el pLDDT promedio (métrica de confianza, 0-100 — más es mejor).

`show_structure` renderiza el PDB en 3D dentro de la notebook con `py3Dmol`. El color del cartoon va por residuo según el orden de la cadena (azul → rojo).


In [ ]:
@torch.no_grad()
def predict_structure(seq):
    inputs = esmfold_tokenizer([seq], return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    output = esmfold_model(**inputs)
    pdb_string = esmfold_model.output_to_pdb(output)[0]
    plddt = float(output["plddt"].mean().item())
    return pdb_string, plddt

def show_structure(pdb_string, title=""):
    view = py3Dmol.view(width=400, height=400)
    view.addModel(pdb_string, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    print(title)
    view.show()


---

### Ejercicio 3.1 — Predecir y comparar estructuras con ESMFold

Para cada proteína en el diccionario `PROTEINS`:

1. Predecir la estructura usando `predict_structure(seq)`.
2. Guardar el PDB resultante y el pLDDT promedio.
3. Visualizar la estructura con `show_structure(pdb, title=nombre)`.

Al final, mostrar una tabla con `[Proteína, Largo (aa), pLDDT promedio]`.


In [ ]:
# === SOLUCIÓN ===
results = {}
for name, seq in PROTEINS.items():
    print(f"Prediciendo {name} ({len(seq)} aa)...")
    pdb, plddt = predict_structure(seq)
    results[name] = {"pdb": pdb, "plddt": plddt, "len": len(seq)}
    print(f"  pLDDT promedio: {plddt:.2f}")


In [ ]:
# === SOLUCIÓN ===
# Visualizar las tres estructuras
for name, r in results.items():
    show_structure(r["pdb"], title=f"{name} — pLDDT {r['plddt']:.1f}")


In [ ]:
# === SOLUCIÓN ===
# Tabla resumen
print(f"{'Proteína':<35} {'Largo':>8} {'pLDDT':>8}")
print("-" * 53)
for name, r in results.items():
    print(f"{name:<35} {r['len']:>8} {r['plddt']:>8.2f}")


### Preguntas conceptuales — Ejercicio 3.1

1. **¿Cuál de las tres proteínas obtuvo mayor pLDDT? ¿Cuál menor?** ¿Te sorprende el resultado?
2. **Mirando las visualizaciones**, ¿qué tipo de estructura secundaria predomina en cada una (hélices, hojas beta, mezcla)?
3. **¿Por qué ESMFold no necesita MSA?** (pista: ¿qué información tiene el modelo "incorporada" después de entrenar sobre 200M secuencias?)
4. **Trade-off ESMFold vs AlphaFold 2**: en qué situación elegirías cada uno.


<!-- SOLUCIÓN -->

**Respuestas — Ejercicio 3.1**

1. **pLDDT comparativo**: las tres deberían dar pLDDT alto (>80) porque son proteínas muy bien representadas en bases de datos (insulina y lisozima son clásicas; GFP fue secuenciada miles de veces). Si alguna fuese una proteína "huérfana" (sin homólogos conocidos), el pLDDT bajaría.

2. **Estructura secundaria visual**: la insulina tiene mayormente hélices alfa con puentes disulfuro. La lisozima es mezcla de hélices y hojas beta. GFP es famosa por su **barril beta** característico (11 hebras beta antiparalelas formando un cilindro que encierra el cromóforo).

3. **Por qué ESMFold no necesita MSA**: porque ESM-2 (el modelo de lenguaje sobre el que está construido) fue entrenado sobre 200M secuencias prediciendo el próximo aminoácido. En ese entrenamiento masivo, el modelo aprende implícitamente las regularidades evolutivas — la información que el MSA aporta explícitamente, ESM la tiene internalizada. Es el mismo principio que los LLMs de lenguaje natural aprenden gramática sin que se las enseñes.

4. **Trade-off ESMFold vs AF2**: ESMFold para escala (predecir cientos de miles de estructuras), proteínas huérfanas (sin homólogos), o cuando no hay tiempo para armar MSA. AF2 para precisión máxima cuando la proteína tiene buena cobertura evolutiva — sigue siendo ligeramente más preciso en GDT_TS, especialmente en regiones difíciles.


---

## Cierre — takeaways

**1. GNN = arquitectura para datos con estructura no-euclidiana.**
Donde el orden de los nodos no importa pero las conexiones sí, message passing es la elección natural. Moléculas, redes sociales, mallas 3D, grafos de conocimiento.

**2. Difusión latente generaliza Stable Diffusion a otros dominios.**
El mismo paradigma sirvió para imágenes (Clase 9), sirve acá para moléculas, y es lo que usa **AlphaFold 3** para estructuras 3D de proteínas y complejos. **Cambia el dominio, no el paradigma.**

**3. La información evolutiva puede internalizarse en un LLM.**
ESMFold demostró que un modelo de lenguaje gigante entrenado sobre 200M secuencias aprende implícitamente lo que el MSA aporta explícitamente. Mismo principio que los LLMs de texto aprenden gramática sin que se las enseñes.

**4. Patrón general en IA para ciencia.**
- Representación correcta (grafos para moléculas, MSA para proteínas, voxels para volúmenes).
- Arquitectura escalable (GNNs, transformers, difusión).
- Datos masivos (ZINC, PDB, AlphaFold DB con 200M estructuras).

**Limitaciones honestas**:
- Modelos generativos no garantizan moléculas sintetizables ni que pasen ensayos in-vitro.
- Validez química ≠ utilidad terapéutica.
- Validación experimental sigue siendo necesaria.
